# Beslisboom

Een simpele beslisboom, met dezelfde folds en maten als `01_baseline.ipynb`.

**LET OP: dit is DUMMY data.** De scores zeggen niets over echte modellen.

## 1. Data inladen

De testfamilies blijven dicht.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.tree import DecisionTreeClassifier, export_text

# De hoofdmap van de repo zoeken (daar staat pyproject.toml) en in sys.path zetten.
HOOFDMAP = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(HOOFDMAP))
DATA_MAP = HOOFDMAP / "outputs" / "candidate"

from src.candidate.data.features import selecteer_features
from src.candidate.data.labelling_rule import laad_benchmark, maak_configuratietabel, voldoende_categorieen

scenarios = pd.read_csv(DATA_MAP / "scenarios.csv")
configuraties = maak_configuratietabel(laad_benchmark(DATA_MAP / "dummy_benchmark.csv"))

with open(DATA_MAP / "split_manifest.json", encoding="utf-8") as bestand:
    manifest = json.load(bestand)

train = scenarios[~scenarios["family_id"].isin(manifest["test_families"])]
len(train)

1672

## 2. Features klaarmaken

Alleen gebruikersinvoer (`features.py`). Een leeg budget wordt 0, `has_energy_budget` laat zien dat er geen budget was. `hardware` wordt one-hot.

In [3]:
def maak_x_y(df):
    x, y = selecteer_features(df)
    x["max_energy_kwh_per_month"] = x["max_energy_kwh_per_month"].fillna(0)
    x = pd.get_dummies(x, columns=["hardware"])
    return x, y


x_train, y_train = maak_x_y(train)
x_train.head()

,input_length_tokens,min_macro_f1,max_latency_ms_per_doc,monthly_volume,max_energy_kwh_per_month,has_energy_budget,hardware_rtx3080,hardware_ryzen5_3600x
0,384,0.842,415.30,123624,0.911700,True,True,False
1,384,0.875,4.43,22447,0.129600,True,False,True
3,384,0.860,222.38,11577,0.000000,False,False,True
4,384,0.818,36.37,1353,0.002304,True,False,True
5,384,0.892,15.15,12923,0.001855,True,False,True


## 3. Trainen per fold

Elke fold is één keer de validatieset. De boom traint op de andere folds. `max_depth=5` houdt de boom klein en leesbaar.

In [4]:
delen = []
for fold in manifest["cv_folds"]:
    in_fold = train["family_id"].isin(fold["families"])

    boom = DecisionTreeClassifier(max_depth=5, random_state=42)
    boom.fit(x_train[~in_fold], y_train[~in_fold])

    validatie = train[in_fold].copy()
    validatie["voorspeld"] = boom.predict(x_train[in_fold])
    delen.append(validatie)

resultaat = pd.concat(delen)
resultaat[["family_id", "label", "voorspeld"]].head()

,family_id,label,voorspeld
6,rtx3080__64,medium,medium
14,rtx3080__64,compact,compact
15,ryzen5_3600x__64,medium,geen
18,rtx3080__64,medium,medium
29,rtx3080__64,compact,compact


## 4. Confusion matrix

Rijen = echt label, kolommen = voorspelling.

In [5]:
labels = ["compact", "medium", "geen"]

matrix = pd.DataFrame(
    confusion_matrix(resultaat["label"], resultaat["voorspeld"], labels=labels),
    index=labels,
    columns=labels,
)
matrix

,compact,medium,geen
compact,468,10,43
medium,8,405,101
geen,90,95,452


## 5. Scores

Dezelfde maten als bij de baseline.

In [6]:
def scores(df):
    gekozen = df[df["voorspeld"] != "geen"]
    violations = sum(
        rij["voorspeld"] not in voldoende_categorieen(configuraties, rij.to_dict())
        for _, rij in gekozen.iterrows()
    )

    kansen = df[df["label"] != "geen"]
    gemiste_kansen = int((kansen["voorspeld"] == "geen").sum())

    return {
        "macro_f1": f1_score(df["label"], df["voorspeld"], labels=labels, average="macro"),
        "violations": violations,
        "keuzes": len(gekozen),
        "violation_rate": violations / len(gekozen),
        "gemiste_kansen": gemiste_kansen,
        "kansen": len(kansen),
        "gemiste_kans_rate": gemiste_kansen / len(kansen),
    }


scores(resultaat)

{'macro_f1': 0.7950907696364827,
 'violations': 193,
 'keuzes': 1076,
 'violation_rate': 0.17936802973977695,
 'gemiste_kansen': 144,
 'kansen': 1035,
 'gemiste_kans_rate': 0.1391304347826087}

## 6. Scores per inputlengte

In [7]:
pd.DataFrame(
    {lengte: scores(deel) for lengte, deel in resultaat.groupby("input_length_tokens")}
).T.round(3)

,macro_f1,violations,keuzes,violation_rate,gemiste_kansen,kansen,gemiste_kans_rate
64,0.756,8.0,220.0,0.036,67.0,279.0,0.240
128,0.870,38.0,247.0,0.154,2.0,213.0,0.009
192,0.833,51.0,267.0,0.191,2.0,222.0,0.009
384,0.761,29.0,155.0,0.187,46.0,174.0,0.264
512,0.700,67.0,187.0,0.358,27.0,147.0,0.184


## 7. De boom bekijken

Eén boom op alle trainfamilies, om de regels te lezen.

In [8]:
boom = DecisionTreeClassifier(max_depth=5, random_state=42)
boom.fit(x_train, y_train)

print(export_text(boom, feature_names=list(x_train.columns)))

|--- min_macro_f1 <= 0.84
|   |--- hardware_rtx3080 <= 0.50
|   |   |--- max_latency_ms_per_doc <= 17.28
|   |   |   |--- input_length_tokens <= 96.00
|   |   |   |   |--- max_latency_ms_per_doc <= 5.61
|   |   |   |   |   |--- class: geen
|   |   |   |   |--- max_latency_ms_per_doc >  5.61
|   |   |   |   |   |--- class: compact
|   |   |   |--- input_length_tokens >  96.00
|   |   |   |   |--- max_latency_ms_per_doc <= 13.12
|   |   |   |   |   |--- class: geen
|   |   |   |   |--- max_latency_ms_per_doc >  13.12
|   |   |   |   |   |--- class: geen
|   |   |--- max_latency_ms_per_doc >  17.28
|   |   |   |--- max_energy_kwh_per_month <= 0.00
|   |   |   |   |--- max_latency_ms_per_doc <= 28.35
|   |   |   |   |   |--- class: compact
|   |   |   |   |--- max_latency_ms_per_doc >  28.35
|   |   |   |   |   |--- class: compact
|   |   |   |--- max_energy_kwh_per_month >  0.00
|   |   |   |   |--- input_length_tokens <= 448.00
|   |   |   |   |   |--- class: compact
|   |   |   |   |---